# 11 — Supervised Contrastive Learning (Phase 2)

**Goal:** Apply Supervised Contrastive Loss (SupCon) to BanglaBERT for more discriminative embeddings.

**Approach (two-stage):**
1. **Stage 1 — Contrastive pre-training:** Fine-tune BanglaBERT encoder with SupCon loss.  
   Same-class embeddings are pulled together, different-class pushed apart.
2. **Stage 2 — Classifier training:** Freeze the encoder, train a linear classifier on top.

**Also tested:** Joint training (SupCon + CE combined in one stage) — simpler and often competitive.

**Backbone:** BanglaBERT (Phase 1 winner)  
**Datasets:** All 3 binary + BanglaSarc3 ternary  
**Baselines:** Plain BanglaBERT (nb05/08), R-Drop+LS (nb10)

In [1]:
!pip install --upgrade transformers huggingface_hub -q

In [2]:
import os, sys, json, time, warnings, gc, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix
)

warnings.filterwarnings('ignore')

# ── Paths ──
if os.path.exists('/kaggle/working'):
    REPO_ROOT = Path('/kaggle/working/Sarcasm_detection')
    BIG_TMP = Path('/kaggle/tmp')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '00_admin').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    BIG_TMP = REPO_ROOT / '_tmp'

PROJECT    = REPO_ROOT
SPLIT_DIR  = PROJECT / '01_data' / 'interim' / 'splits'
TABLE_DIR  = PROJECT / '04_outputs' / 'tables'
PRED_DIR   = PROJECT / '04_outputs' / 'predictions'
LOG_DIR    = PROJECT / '04_outputs' / 'run_logs'
REPORT_DIR = PROJECT / '04_outputs' / 'reports'

for d in [TABLE_DIR, PRED_DIR, LOG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR   = BIG_TMP / 'hf_cache'
TEMP_TRAIN_DIR = BIG_TMP / 'train_cache'
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)

def disk_free_gb(path='/'):
    try: return shutil.disk_usage(path).free / 1e9
    except: return shutil.disk_usage('.').free / 1e9

def clear_hf_cache():
    if HF_CACHE_DIR.exists():
        shutil.rmtree(HF_CACHE_DIR, ignore_errors=True)
        HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project: {PROJECT}")
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  {props.name} — {props.total_memory / 1e9:.1f} GB")
if os.path.exists('/kaggle/working'):
    print(f"Disk (working): {disk_free_gb('/kaggle/working'):.1f} GB")
    print(f"Disk (tmp): {disk_free_gb(BIG_TMP):.1f} GB")
    clear_hf_cache()

Project: /kaggle/working/Sarcasm_detection
GPU: True
  Tesla T4 — 15.6 GB
Disk (working): 20.9 GB
Disk (tmp): 1310.3 GB


In [3]:
# ── Configuration ──

MODEL_NAME = 'csebuetnlp/banglabert'
MODEL_TAG  = 'banglabert'

DATASETS = {
    'ben_sarc_binary': {
        'train': SPLIT_DIR / 'ben_sarc_binary_train.csv',
        'val':   SPLIT_DIR / 'ben_sarc_binary_val.csv',
        'test':  SPLIT_DIR / 'ben_sarc_binary_test.csv',
        'num_labels': 2, 'label_col': 'label_binary',
    },
    'banglasarc_binary': {
        'train': SPLIT_DIR / 'banglasarc_binary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc_binary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc_binary_test.csv',
        'num_labels': 2, 'label_col': 'label_binary',
    },
    'banglasarc3_binary': {
        'train': SPLIT_DIR / 'banglasarc3_binary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc3_binary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc3_binary_test.csv',
        'num_labels': 2, 'label_col': 'label_binary',
    },
    'banglasarc3_ternary': {
        'train': SPLIT_DIR / 'banglasarc3_ternary_train.csv',
        'val':   SPLIT_DIR / 'banglasarc3_ternary_val.csv',
        'test':  SPLIT_DIR / 'banglasarc3_ternary_test.csv',
        'num_labels': 3, 'label_col': 'label_original',
    },
}

LABEL_MAP = {'Non-Sarcastic': 0, 'Neutral': 1, 'Sarcastic': 2}

TECHNIQUES = {
    'supcon_joint': {'supcon_weight': 0.5},
    'supcon_twostage': {'supcon_weight': 1.0},
}

MAX_LENGTH    = 128
EPOCHS        = 3
BATCH_SIZE    = 4                     # reduced from 8
GRAD_ACCUM    = 2                     # effective batch = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1
SEED          = 42
SUPCON_TEMP   = 0.07
TEXT_COL      = 'text'

total_runs = len(TECHNIQUES) * len(DATASETS)
print(f"Techniques: {list(TECHNIQUES.keys())}")
print(f"Datasets: {list(DATASETS.keys())}")
print(f"Total runs: {total_runs}")

Techniques: ['supcon_joint', 'supcon_twostage']
Datasets: ['ben_sarc_binary', 'banglasarc_binary', 'banglasarc3_binary', 'banglasarc3_ternary']
Total runs: 8


In [4]:
# ── Dataset & helpers ──

class SarcasmDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length',
                                   max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def load_split(csv_path, label_col):
    df = pd.read_csv(csv_path)
    texts = df[TEXT_COL].astype(str).tolist()
    if df[label_col].dtype == object:
        labels = df[label_col].map(LABEL_MAP).astype(int).tolist()
    else:
        labels = df[label_col].astype(int).tolist()
    return texts, labels


def compute_all_metrics(labels, preds, num_labels):
    result = {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }
    if num_labels == 2:
        result['binary_f1'] = f1_score(labels, preds, average='binary', pos_label=1)
        result['precision'] = precision_score(labels, preds, average='binary', pos_label=1)
        result['recall'] = recall_score(labels, preds, average='binary', pos_label=1)
    else:
        result['macro_precision'] = precision_score(labels, preds, average='macro')
        result['macro_recall'] = recall_score(labels, preds, average='macro')
    return result


# Sanity check
for ds_name, ds_cfg in DATASETS.items():
    tr, trl = load_split(ds_cfg['train'], ds_cfg['label_col'])
    print(f"{ds_name}: train={len(tr)}, labels={sorted(set(trl))}")

ben_sarc_binary: train=20508, labels=[0, 1]
banglasarc_binary: train=4089, labels=[0, 1]


banglasarc3_binary: train=6413, labels=[0, 1]


banglasarc3_ternary: train=9657, labels=[0, 1, 2]


In [5]:
# ── Supervised Contrastive Loss ──

class SupConLoss(nn.Module):
    """Supervised Contrastive Loss (Khosla et al., NeurIPS 2020)."""
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        """
        Args:
            features: [batch_size, hidden_dim] — L2-normalized embeddings
            labels: [batch_size] — integer class labels
        """
        device = features.device
        batch_size = features.shape[0]

        # Similarity matrix
        sim = torch.matmul(features, features.T) / self.temperature  # [B, B]

        # Mask: 1 where same class (excluding self)
        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)  # [B, B]
        self_mask = torch.eye(batch_size, device=device)
        mask = mask - self_mask  # Remove self-pairs

        # For numerical stability
        logits_max, _ = sim.max(dim=1, keepdim=True)
        sim = sim - logits_max.detach()

        # Denominator: all pairs except self
        exp_sim = torch.exp(sim) * (1 - self_mask)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)

        # Mean log-likelihood over positive pairs
        pos_count = mask.sum(dim=1)
        # Avoid division by zero for samples with no same-class partners in batch
        pos_count = torch.clamp(pos_count, min=1)
        mean_log_prob = (mask * log_prob).sum(dim=1) / pos_count

        loss = -mean_log_prob.mean()
        return loss

In [6]:
# ── Joint SupCon+CE Trainer (memory efficient) ──
class SupConJointTrainer(Trainer):
    """
    Joint training: CE loss + supcon_weight * SupCon loss.
    Memory efficient: does NOT store all hidden layers.
    Gradients for SupCon flow to the encoder via a second forward pass.
    Handles DataParallel wrapper automatically.
    """
    def __init__(self, *args, supcon_weight=0.5, temperature=0.07, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.supcon_weight = supcon_weight
        self.supcon_loss_fn = SupConLoss(temperature=temperature)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')

        # 1. Standard forward for classification (no hidden_states)
        outputs = model(**inputs)
        logits = outputs.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        ce_loss = F.cross_entropy(logits, labels, weight=w)

        # 2. Get underlying model (handles DataParallel)
        unwrapped = model.module if hasattr(model, 'module') else model
        base_outputs = unwrapped.base_model(**inputs)   # recomputed, gradients flow
        cls_hidden = base_outputs.last_hidden_state[:, 0, :]
        cls_norm = F.normalize(cls_hidden, p=2, dim=1)

        # 3. SupCon loss
        supcon_loss = self.supcon_loss_fn(cls_norm, labels)

        loss = ce_loss + self.supcon_weight * supcon_loss
        return (loss, outputs) if return_outputs else loss

In [7]:
# ── Two-stage SupCon functions ──

def run_twostage(dataset_tag, dataset_cfg, tokenizer):
    """
    Stage 1: Fine-tune encoder with SupCon loss (no classification head).
    Stage 2: Freeze encoder, train a linear classifier with CE loss.
    """
    num_labels = dataset_cfg['num_labels']
    label_col = dataset_cfg['label_col']

    tr_texts, tr_labels = load_split(dataset_cfg['train'], label_col)
    vl_texts, vl_labels = load_split(dataset_cfg['val'], label_col)
    te_texts, te_labels = load_split(dataset_cfg['test'], label_col)

    train_ds = SarcasmDataset(tr_texts, tr_labels, tokenizer, MAX_LENGTH)
    val_ds   = SarcasmDataset(vl_texts, vl_labels, tokenizer, MAX_LENGTH)
    test_ds  = SarcasmDataset(te_texts, te_labels, tokenizer, MAX_LENGTH)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ─── STAGE 1: Contrastive encoder training ───
    print("  Stage 1: Contrastive encoder training...")
    encoder = AutoModel.from_pretrained(MODEL_NAME, cache_dir=str(HF_CACHE_DIR)).to(device)
    encoder.train()

    supcon_loss_fn = SupConLoss(temperature=SUPCON_TEMP)
    optimizer = torch.optim.AdamW(encoder.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    total_steps = len(train_loader) * 2  # 2 epochs for stage 1
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * WARMUP_RATIO), num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    for epoch in range(2):
        epoch_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            if scaler:
                with torch.amp.autocast('cuda'):
                    outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)
                    cls_emb = outputs.last_hidden_state[:, 0, :]
                    cls_norm = F.normalize(cls_emb, p=2, dim=1)
                    loss = supcon_loss_fn(cls_norm, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)
                cls_emb = outputs.last_hidden_state[:, 0, :]
                cls_norm = F.normalize(cls_emb, p=2, dim=1)
                loss = supcon_loss_fn(cls_norm, labels)
                loss.backward()
                optimizer.step()

            scheduler.step()
            epoch_loss += loss.item()

        print(f"    Epoch {epoch+1}/2 — SupCon loss: {epoch_loss / len(train_loader):.4f}")

    # ─── STAGE 2: Freeze encoder, train classifier ───
    print("  Stage 2: Training classifier on frozen encoder...")
    encoder.eval()

    # Extract embeddings
    def extract_embeddings(dataset):
        loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)
        all_embs, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                if torch.cuda.is_available():
                    with torch.amp.autocast('cuda'):
                        out = encoder(input_ids=input_ids, attention_mask=attention_mask)
                else:
                    out = encoder(input_ids=input_ids, attention_mask=attention_mask)
                cls_emb = out.last_hidden_state[:, 0, :]
                all_embs.append(cls_emb.cpu())
                all_labels.append(batch['labels'])
        return torch.cat(all_embs), torch.cat(all_labels)

    tr_embs, tr_labs = extract_embeddings(train_ds)
    vl_embs, vl_labs = extract_embeddings(val_ds)
    te_embs, te_labs = extract_embeddings(test_ds)

    hidden_dim = tr_embs.shape[1]

    # Simple linear classifier
    classifier = nn.Linear(hidden_dim, num_labels).to(device)
    cls_optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=0.01)

    # Class weights for ternary
    cls_weight = None
    if num_labels == 3:
        counts = Counter(tr_labs.numpy().tolist())
        total = sum(counts.values())
        cls_weight = torch.tensor(
            [total / (num_labels * counts[i]) for i in range(num_labels)],
            dtype=torch.float32
        ).to(device)

    # Train classifier for 20 epochs (fast — just a linear layer)
    best_val_f1 = 0
    best_state = None
    for ep in range(20):
        classifier.train()
        idx = torch.randperm(tr_embs.shape[0])
        for start in range(0, len(idx), 64):
            batch_idx = idx[start:start+64]
            embs = tr_embs[batch_idx].to(device)
            labs = tr_labs[batch_idx].to(device)
            cls_optimizer.zero_grad()
            logits = classifier(embs)
            loss = F.cross_entropy(logits, labs, weight=cls_weight)
            loss.backward()
            cls_optimizer.step()

        # Validate
        classifier.eval()
        with torch.no_grad():
            vl_logits = classifier(vl_embs.to(device))
            vl_preds = vl_logits.argmax(dim=1).cpu().numpy()
            vl_f1 = f1_score(vl_labs.numpy(), vl_preds, average='macro')
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_state = {k: v.clone() for k, v in classifier.state_dict().items()}

    classifier.load_state_dict(best_state)
    print(f"    Best val Macro-F1: {best_val_f1:.4f}")

    # ─── Predict ───
    classifier.eval()
    with torch.no_grad():
        # Val
        vl_logits = classifier(vl_embs.to(device))
        vl_probs = F.softmax(vl_logits, dim=1).cpu().numpy()
        vl_preds = vl_logits.argmax(dim=1).cpu().numpy()
        # Test
        te_logits = classifier(te_embs.to(device))
        te_probs = F.softmax(te_logits, dim=1).cpu().numpy()
        te_preds = te_logits.argmax(dim=1).cpu().numpy()

    # Cleanup encoder
    del encoder, classifier, tr_embs, vl_embs, te_embs
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return (
        vl_preds, vl_probs, vl_labels,
        te_preds, te_probs, te_labels,
        vl_texts, te_texts,
    )

In [8]:
# ── HF metrics function for joint trainer ──

def make_compute_metrics(num_labels):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        result = {
            'accuracy': accuracy_score(labels, preds),
            'macro_f1': f1_score(labels, preds, average='macro'),
            'weighted_f1': f1_score(labels, preds, average='weighted'),
        }
        if num_labels == 2:
            result['binary_f1'] = f1_score(labels, preds, average='binary', pos_label=1)
            result['precision'] = precision_score(labels, preds, average='binary', pos_label=1)
            result['recall'] = recall_score(labels, preds, average='binary', pos_label=1)
        return result
    return compute_metrics

In [9]:
# ── Main training function ──

def train_supcon(technique_tag, technique_cfg, dataset_tag, dataset_cfg):
    print(f"\n{'='*70}")
    print(f"  {technique_tag} x {dataset_tag}")
    if os.path.exists('/kaggle/working'):
        print(f"  Disk (tmp): {disk_free_gb(BIG_TMP):.1f} GB | working: {disk_free_gb('/kaggle/working'):.1f} GB")
    print(f"{'='*70}")
    t_start = time.time()

    num_labels = dataset_cfg['num_labels']
    label_col = dataset_cfg['label_col']

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=str(HF_CACHE_DIR))

    if technique_tag == 'supcon_twostage':
        # ── Two-stage approach (unchanged) ──
        (
            vl_preds, vl_probs, vl_labels,
            te_preds, te_probs, te_labels,
            vl_texts, te_texts,
        ) = run_twostage(dataset_tag, dataset_cfg, tokenizer)

        val_metrics = compute_all_metrics(vl_labels, vl_preds, num_labels)
        test_metrics = compute_all_metrics(te_labels, te_preds, num_labels)

    else:
        # ── Joint approach with memory optimisations ──
        tr_texts, tr_labels = load_split(dataset_cfg['train'], label_col)
        vl_texts, vl_labels = load_split(dataset_cfg['val'], label_col)
        te_texts, te_labels = load_split(dataset_cfg['test'], label_col)

        train_ds = SarcasmDataset(tr_texts, tr_labels, tokenizer, MAX_LENGTH)
        val_ds   = SarcasmDataset(vl_texts, vl_labels, tokenizer, MAX_LENGTH)
        test_ds  = SarcasmDataset(te_texts, te_labels, tokenizer, MAX_LENGTH)

        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=num_labels, ignore_mismatched_sizes=True,
            cache_dir=str(HF_CACHE_DIR)
        )
        # Enable gradient checkpointing for extra memory saving
        model.gradient_checkpointing_enable()

        # Class weights
        class_weights = None
        if num_labels == 3:
            counts = Counter(tr_labels)
            total = sum(counts.values())
            class_weights = torch.tensor(
                [total / (num_labels * counts[i]) for i in range(num_labels)],
                dtype=torch.float32
            )

        run_tmp = TEMP_TRAIN_DIR / f"supcon_{dataset_tag}"
        if run_tmp.exists(): shutil.rmtree(run_tmp)
        run_tmp.mkdir(parents=True, exist_ok=True)

        training_args = TrainingArguments(
            output_dir=str(run_tmp),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,       # 8
            gradient_accumulation_steps=GRAD_ACCUM,          # 2 → effective batch 8
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            warmup_ratio=WARMUP_RATIO,
            eval_strategy='epoch',
            save_strategy='epoch',
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model='macro_f1',
            greater_is_better=True,
            logging_steps=100,
            fp16=torch.cuda.is_available(),
            seed=SEED,
            report_to='none',
            dataloader_num_workers=2,
        )

        trainer = SupConJointTrainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=make_compute_metrics(num_labels),
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
            supcon_weight=technique_cfg['supcon_weight'],
            temperature=SUPCON_TEMP,
            class_weights=class_weights,
        )

        trainer.train()

        val_out = trainer.predict(val_ds)
        vl_preds = np.argmax(val_out.predictions, axis=-1)
        vl_probs = torch.softmax(torch.tensor(val_out.predictions), dim=-1).numpy()
        val_metrics = {k.replace('test_', ''): v for k, v in val_out.metrics.items()}

        test_out = trainer.predict(test_ds)
        te_preds = np.argmax(test_out.predictions, axis=-1)
        te_probs = torch.softmax(torch.tensor(test_out.predictions), dim=-1).numpy()
        test_metrics = {k.replace('test_', ''): v for k, v in test_out.metrics.items()}

        del model, trainer, train_ds, val_ds, test_ds
        if run_tmp.exists(): shutil.rmtree(run_tmp)

    print(f"  Val  — Acc: {val_metrics['accuracy']:.4f} | Macro-F1: {val_metrics['macro_f1']:.4f}")
    print(f"  Test — Acc: {test_metrics['accuracy']:.4f} | Macro-F1: {test_metrics['macro_f1']:.4f}")

    # Save predictions
    for split_name, texts, labels, preds, probs in [
        ('test', te_texts, te_labels, te_preds, te_probs),
        ('val',  vl_texts, vl_labels, vl_preds, vl_probs),
    ]:
        pred_dict = {'text': texts, 'true_label': labels, 'pred_label': preds}
        for c in range(num_labels):
            pred_dict[f'prob_{c}'] = probs[:, c]
        pd.DataFrame(pred_dict).to_csv(
            PRED_DIR / f"11_{technique_tag}_{dataset_tag}_{split_name}_predictions.csv",
            index=False
        )

    cm = confusion_matrix(te_labels, te_preds).tolist()
    t_elapsed = time.time() - t_start

    result = {
        'technique': technique_tag,
        'model_tag': MODEL_TAG,
        'dataset': dataset_tag,
        'num_labels': num_labels,
        'supcon_weight': technique_cfg['supcon_weight'],
        'val_accuracy':  val_metrics['accuracy'],
        'val_macro_f1':  val_metrics['macro_f1'],
        'test_accuracy':  test_metrics['accuracy'],
        'test_macro_f1':  test_metrics['macro_f1'],
        'test_weighted_f1': test_metrics['weighted_f1'],
        'confusion_matrix': json.dumps(cm),
        'time_seconds': round(t_elapsed, 1),
    }

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    clear_hf_cache()
    print(f"  Done in {t_elapsed/60:.1f} min")

    return result

In [10]:
# ── Run all experiments ──

all_results = []
summary_path = TABLE_DIR / '11_supcon_summary.csv'

if summary_path.exists():
    prev_df = pd.read_csv(summary_path)
    all_results = prev_df.to_dict('records')
    done_keys = set(zip(prev_df['technique'], prev_df['dataset']))
    print(f"Resuming: {len(done_keys)} runs done")
else:
    done_keys = set()

if TEMP_TRAIN_DIR.exists():
    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
    TEMP_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
clear_hf_cache()

run_num = len(done_keys)

for tech_tag, tech_cfg in TECHNIQUES.items():
    for ds_tag, ds_cfg in DATASETS.items():
        if (tech_tag, ds_tag) in done_keys:
            print(f"Skipping {tech_tag} x {ds_tag} (done)")
            continue

        run_num += 1
        print(f"\n>>> Run {run_num}/{total_runs}")

        if os.path.exists('/kaggle/working') and disk_free_gb('/kaggle/working') < 3.0:
            clear_hf_cache()

        # Reset GPU memory stats
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.empty_cache()

        try:
            result = train_supcon(tech_tag, tech_cfg, ds_tag, ds_cfg)
            all_results.append(result)
            pd.DataFrame(all_results).to_csv(summary_path, index=False)
            print(f"  Saved. {total_runs - run_num} remaining.")
        except Exception as e:
            print(f"  FAILED: {e}")
            if all_results:
                pd.DataFrame(all_results).to_csv(summary_path, index=False)
            if TEMP_TRAIN_DIR.exists():
                shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
            clear_hf_cache()
            raise

if TEMP_TRAIN_DIR.exists():
    shutil.rmtree(TEMP_TRAIN_DIR, ignore_errors=True)
clear_hf_cache()

print(f"\n{'='*70}")
print(f"All {total_runs} runs complete!")


>>> Run 1/8

  supcon_joint x ben_sarc_binary
  Disk (tmp): 1310.3 GB | working: 20.9 GB


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

    Epoch 1/2 — SupCon loss: 0.7985


    Epoch 2/2 — SupCon loss: 0.7746
  Stage 2: Training classifier on frozen encoder...


    Best val Macro-F1: 0.4091


  Val  — Acc: 0.4308 | Macro-F1: 0.4091
  Test — Acc: 0.4338 | Macro-F1: 0.4140


  Done in 6.9 min
  Saved. 0 remaining.

All 8 runs complete!


In [11]:
# ── Results ──

results_df = pd.read_csv(summary_path)

print("="*90)
print("  SUPCON RESULTS (Test Macro-F1)")
print("="*90)
print(results_df[['technique', 'dataset', 'test_accuracy', 'test_macro_f1', 'test_weighted_f1', 'time_seconds']]
      .to_string(index=False, float_format='%.4f'))

  SUPCON RESULTS (Test Macro-F1)
      technique             dataset  test_accuracy  test_macro_f1  test_weighted_f1  time_seconds
   supcon_joint     ben_sarc_binary         0.8027         0.8026            0.8026     3727.0000
   supcon_joint   banglasarc_binary         0.9844         0.9834            0.9843      753.9000
   supcon_joint  banglasarc3_binary         0.7606         0.7606            0.7606     1176.0000
   supcon_joint banglasarc3_ternary         0.6548         0.6537            0.6538     1762.6000
supcon_twostage     ben_sarc_binary         0.5000         0.3333            0.3333      863.9000
supcon_twostage   banglasarc_binary         0.9824         0.9813            0.9824      178.1000
supcon_twostage  banglasarc3_binary         0.7294         0.7294            0.7294      275.6000
supcon_twostage banglasarc3_ternary         0.4338         0.4140            0.4144      411.4000


In [12]:
# ── Pivot with baselines ──

pivot = results_df.pivot_table(
    index='technique', columns='dataset',
    values='test_macro_f1', aggfunc='first'
)

# Add baselines
bb_path = TABLE_DIR / '05_baseline_banglabert_binary_summary_all_datasets.csv'
if bb_path.exists():
    bb_df = pd.read_csv(bb_path)
    bb_row = {}
    for _, r in bb_df.iterrows():
        ds = r.get('dataset') or r.get('dataset_tag') or r.get('dataset_name')
        f1 = r.get('test_macro_f1') or r.get('macro_f1')
        if ds and f1: bb_row[ds] = f1
    if bb_row: pivot.loc['plain_baseline (nb05)'] = bb_row

# Add nb10 best if available
nb10_path = TABLE_DIR / '10_rdrop_ls_summary.csv'
if nb10_path.exists():
    nb10_df = pd.read_csv(nb10_path)
    # Find best technique per dataset
    nb10_best = nb10_df.loc[nb10_df.groupby('dataset')['test_macro_f1'].idxmax()]
    best_row = {}
    for _, r in nb10_best.iterrows():
        best_row[r['dataset']] = r['test_macro_f1']
    if best_row:
        pivot.loc['best_rdrop_ls (nb10)'] = best_row

# Ternary baseline
tern_path = TABLE_DIR / '08_multi_backbone_ternary_summary.csv'
if tern_path.exists():
    tern_df = pd.read_csv(tern_path)
    bb_tern = tern_df[tern_df['model_tag'] == 'banglabert']
    if len(bb_tern) > 0 and 'plain_baseline (nb05)' in pivot.index:
        pivot.loc['plain_baseline (nb05)']['banglasarc3_ternary'] = bb_tern.iloc[0]['test_macro_f1']

pivot['mean'] = pivot.mean(axis=1)
pivot = pivot.sort_values('mean', ascending=False)

print("="*90)
print("  MACRO-F1 COMPARISON — SupCon vs Baselines vs R-Drop/LS")
print("="*90)
print(pivot.to_string(float_format='%.4f'))
print(f"\nBest overall: {pivot['mean'].idxmax()} (mean={pivot['mean'].max():.4f})")

  MACRO-F1 COMPARISON — SupCon vs Baselines vs R-Drop/LS
dataset                banglasarc3_binary  banglasarc3_ternary  banglasarc_binary  ben_sarc_binary   mean
technique                                                                                                
plain_baseline (nb05)              0.7666                  NaN             0.9834           0.8018 0.8506
supcon_joint                       0.7606               0.6537             0.9834           0.8026 0.8001
best_rdrop_ls (nb10)               0.7443               0.6578             0.9813           0.8049 0.7971
supcon_twostage                    0.7294               0.4140             0.9813           0.3333 0.6145

Best overall: plain_baseline (nb05) (mean=0.8506)


In [13]:
# ── Save artifacts ──

cm_dict = {}
for _, row in results_df.iterrows():
    cm_dict[f"{row['technique']}_{row['dataset']}"] = json.loads(row['confusion_matrix'])
with open(TABLE_DIR / '11_supcon_confusion_matrices.json', 'w') as f:
    json.dump(cm_dict, f, indent=2)

pivot.to_csv(TABLE_DIR / '11_supcon_macro_f1_pivot.csv')

results_df[['technique', 'dataset', 'num_labels', 'supcon_weight', 'time_seconds']].to_csv(
    LOG_DIR / '11_supcon_run_metadata.csv', index=False
)

print("All artifacts saved.")
print("\nFiles produced:")
for p in sorted(PROJECT.rglob('11_*')):
    if p.is_file():
        print(f"  {p.relative_to(PROJECT)}  ({p.stat().st_size / 1e3:.1f} KB)")

All artifacts saved.

Files produced:
  02_notebooks/11_supcon_learning.ipynb  (120.8 KB)
  04_outputs/predictions/11_supcon_joint_banglasarc3_binary_test_predictions.csv  (179.7 KB)
  04_outputs/predictions/11_supcon_joint_banglasarc3_binary_val_predictions.csv  (179.4 KB)
  04_outputs/predictions/11_supcon_joint_banglasarc3_ternary_test_predictions.csv  (297.8 KB)
  04_outputs/predictions/11_supcon_joint_banglasarc3_ternary_val_predictions.csv  (292.0 KB)
  04_outputs/predictions/11_supcon_joint_banglasarc_binary_test_predictions.csv  (109.1 KB)
  04_outputs/predictions/11_supcon_joint_banglasarc_binary_val_predictions.csv  (103.1 KB)
  04_outputs/predictions/11_supcon_joint_ben_sarc_binary_test_predictions.csv  (606.6 KB)
  04_outputs/predictions/11_supcon_joint_ben_sarc_binary_val_predictions.csv  (584.7 KB)
  04_outputs/predictions/11_supcon_twostage_banglasarc3_binary_test_predictions.csv  (179.7 KB)
  04_outputs/predictions/11_supcon_twostage_banglasarc3_binary_val_predictions.c